# Starter Kit FahMai RAG

This notebook walks through building a minimalist **Retrieval-Augmented Generation (RAG)** system for the FahMai electronics store Q&A challenge.

**Sections:**
- **Section 0:** Setup & LLM Test
- **Section 1:** Dense Retrieval (MiniLM)
- **Section 2:** Sparse Retrieval (BM25)
- **Section 3:** Hybrid Retrieval (RRF)
- **Section 4:** What's Next?

In [1]:
from google.colab import files
import os

# 1. อัปโหลดไฟล์ kaggle.json
if not os.path.exists('/content/kaggle.json'):
    uploaded = files.upload()

# 2. จัดการย้ายไฟล์ไปที่ ~/.kaggle/ และตั้งค่าสิทธิ์
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [2]:
# 3. ดาวน์โหลด Dataset (ระบุชื่อการแข่งขันตามที่คุณให้มา)
!kaggle competitions download -c super-ai-engineer-s-6-fah-mai-rag-challenge-level-1

# 4. แตกไฟล์ Zip (ไฟล์มักจะชื่อตามการแข่งขัน)
import zipfile
import os

zip_file = 'super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip'

if os.path.exists(zip_file):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall('dataset')
    print("แตกไฟล์เรียบร้อยแล้วในโฟลเดอร์ 'dataset'")
else:
    print("ไม่พบไฟล์ zip กรุณาตรวจสอบชื่อไฟล์จากการดาวน์โหลด")

100% 321k/321k [00:00<00:00, 95.8MB/s]

แตกไฟล์เรียบร้อยแล้วในโฟลเดอร์ 'dataset'


In [ ]:
# Load the data from kaggle and upload here.
!unzip data.zip

In [3]:
# === CONFIGURATION ===
# Change N_QUESTIONS to 100 for a full competition run.

N_QUESTIONS = 10  # 10 for demo, 100 for real submission
DATA_DIR = "/content/dataset/data"
KB_DIR = f"{DATA_DIR}/knowledge_base"

---

## Section 0: Setup & LLM Test

First, install dependencies and test the ThaiLLM API — **without** any retrieval. This shows why RAG is needed.

ThaiLLM website: https://playground.thaillm.or.th/chat/

In [4]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 68.5 MB/s eta 0:00:00


In [ ]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
from google.colab import userdata

THAILLM_API_KEY = userdata.get('ThaiLLM')

In [ ]:
def ask_llm(messages, model="typhoon", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg
    """
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            # Rate limit — wait and retry
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None


def parse_answer(text):
    """Extract answer number from LLM response."""
    if text is None:
        return None
    # Remove any <think>...</think> blocks (some models do chain-of-thought)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Look for ANSWER: X pattern
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        return int(m.group(1))
    # Fallback: first standalone number 1-10
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10:
            return int(d)
    return None

### Test the LLM without RAG

Let's ask a FahMai-specific question *without* any context. The LLM shouldn't know the answer.

In [ ]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

### Load Questions

In [ ]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions, using first {N_QUESTIONS} for this run")
print(f"\nExample — Q1: {questions[0]['question']}")
for k, v in questions[0]["choices"].items():
    print(f"  {k}. {v}")

---
## Section 1: Dense Retrieval (MiniLM)

**Dense retrieval** converts text into vectors (embeddings) and finds relevant chunks by cosine similarity.

The pipeline: **Load docs → Chunk → Embed → Retrieve → Generate**

### 1.1 Load Knowledge Base

In [ ]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

### 1.2 Chunking

LLMs have limited context windows, and long documents dilute relevance. We split each document into smaller **chunks** using a fixed-size sliding window with overlap.

In [ ]:
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128

def make_chunks(text, size, overlap):
    """Split text into overlapping fixed-size windows."""
    if len(text) <= size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return chunks

# Build all chunks
chunks = []
for doc in documents:
    for window in make_chunks(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP):
        chunks.append({"text": window, "source": doc["path"]})

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\n--- Sample chunk ---")
print(f"Source: {chunks[0]['source']}")
print(chunks[0]["text"][:300])

### 1.3 Embedding

We use `paraphrase-multilingual-MiniLM-L12-v2` — a small (118M params), multilingual embedding model that produces 384-dimensional vectors. It supports Thai out of the box and doesn't require any special prefixes.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Embed all chunks
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")  # (n_chunks, 384)

### 1.4 Retrieve

Embed the question, then find the most similar chunks via dot product (= cosine similarity for normalized vectors).

In [ ]:
TOP_K = 5

def dense_retrieve(query, chunk_embs, k=TOP_K):
    """Return indices of top-k most similar chunks to the query."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()  # cosine similarity (vectors are normalized)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Demo: retrieve for Q1
q = questions[0]
idx, scores = dense_retrieve(q["question"], chunk_embeddings)

print(f"Question: {q['question']}\n")
for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"  Rank {rank} (score={s:.3f}) [{chunks[i]['source']}]")
    print(f"  {chunks[i]['text'][:150]}...")
    print()

### 1.5 Generate Answer

Send the retrieved context + question + choices to the LLM and parse the answer.

In [ ]:
# A very basic system prompt — you should improve this!
SYSTEM_PROMPT = "ตอบคำถามจากข้อมูลที่ให้มา เลือกตัวเลือกที่ถูกต้องที่สุด ตอบเป็น ANSWER: X เท่านั้น"

def build_rag_prompt(question, choices, retrieved_chunks):
    """Build the user prompt with retrieved context.

    TODO: Design your own prompt format.
    Think about: How should you present the context? The choices?
    What instructions help the LLM pick the right answer?
    """
    context = "\n\n".join(
        f"--- Chunk {i+1} ---\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    choices_text = "\n".join(f"{k}. {v}" for k, v in choices.items())
    # === CUSTOMIZE THIS PROMPT ===
    return (
        f"{context}\n\n"
        f"คำถาม: {question}\n\n"
        f"ตัวเลือก:\n{choices_text}\n\n"
        f"ตอบ ANSWER: X (X คือหมายเลขตัวเลือก 1-10)"
    )

# Demo: answer Q1
q = questions[0]
idx, _ = dense_retrieve(q["question"], chunk_embeddings)
retrieved = [chunks[i] for i in idx]

prompt = build_rag_prompt(q["question"], q["choices"], retrieved)
raw = ask_llm([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
])
answer = parse_answer(raw)
print(f"Q{q['id']}: {q['question']}")
print(f"LLM raw: {raw}")
print(f"Parsed answer: {answer}")

In [ ]:
retrieved

### 1.6 Run All Questions (Dense)

Loop through questions, retrieve, generate, and collect predictions.

In [ ]:
def run_pipeline(questions, retrieve_fn, label="dense", n=N_QUESTIONS):
    """Run retrieval + LLM on first n questions. Returns predictions dict."""
    predictions = {}
    for i, q in enumerate(questions[:n]):
        retrieved_chunks = retrieve_fn(q["question"])
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)
        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ])
        pred = parse_answer(raw)
        predictions[q["id"]] = pred if pred else 1  # default to 1 if parse fails
        print(f"  Q{q['id']:>3}: pred={predictions[q['id']]}")
        time.sleep(0.3)  # be nice to the API
    print(f"\n{label}: answered {len(predictions)} questions")
    return predictions

# Dense retrieval function
def dense_retrieve_chunks(query):
    idx, _ = dense_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

dense_preds = run_pipeline(questions, dense_retrieve_chunks, label="dense")

---
## Section 2: Sparse Retrieval (BM25)

**BM25** is a keyword-matching algorithm. It scores documents by how many query terms they contain, weighted by term rarity (IDF). No neural network needed.

### 2.1 Thai Tokenization

BM25 needs tokenized text. Thai has no spaces between words, so we use `pythainlp` to segment.

In [ ]:
from pythainlp.tokenize import word_tokenize

# Demo: tokenize a Thai sentence
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"
tokens = word_tokenize(sample, engine="newmm")
print(f"Input:  {sample}")
print(f"Tokens: {tokens}")

### 2.2 Build BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize all chunks
tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks")

### 2.3 Retrieve with BM25

Compare BM25 results with dense results for the same question.

In [ ]:
def bm25_retrieve(query, k=TOP_K):
    """Return top-k chunk indices using BM25."""
    tokens = word_tokenize(query, engine="newmm")
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Compare: same question, two retrieval methods
q = questions[0]
print(f"Question: {q['question']}\n")

d_idx, _ = dense_retrieve(q["question"], chunk_embeddings)
b_idx, _ = bm25_retrieve(q["question"])

print("Dense top-5 sources:")
for i in d_idx:
    print(f"  {chunks[i]['source']}")

print("\nBM25 top-5 sources:")
for i in b_idx:
    print(f"  {chunks[i]['source']}")

### 2.4 Run All Questions (BM25)

In [ ]:
def bm25_retrieve_chunks(query):
    idx, _ = bm25_retrieve(query)
    return [chunks[i] for i in idx]

bm25_preds = run_pipeline(questions, bm25_retrieve_chunks, label="bm25")

---
## Section 3: Hybrid Retrieval (RRF)

**Hybrid** combines dense and sparse results. The idea: dense is good at semantic matching, BM25 is good at exact keyword matching. Together they cover more cases.

We use **Reciprocal Rank Fusion (RRF)** to merge the two ranked lists:

$$\text{score}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60). Documents ranked highly by *either* method get a high combined score.

In [ ]:
def hybrid_retrieve(query, chunk_embs, k=TOP_K, rrf_k=60):
    """Combine dense + BM25 results using Reciprocal Rank Fusion."""
    # Get top candidates from each method (fetch more than k to improve fusion)
    fetch_k = k * 2
    d_idx, _ = dense_retrieve(query, chunk_embs, k=fetch_k)
    b_idx, _ = bm25_retrieve(query, k=fetch_k)

    # Compute RRF scores
    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    # Sort by combined score, return top-k
    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx

# Demo
q = questions[0]
h_idx = hybrid_retrieve(q["question"], chunk_embeddings)
print(f"Question: {q['question']}\n")
print("Hybrid top-5 sources:")
for i in h_idx:
    print(f"  {chunks[i]['source']}")

### 3.2 Run All Questions (Hybrid)

In [ ]:
def hybrid_retrieve_chunks(query):
    idx = hybrid_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

hybrid_preds = run_pipeline(questions, hybrid_retrieve_chunks, label="hybrid")

### 3.3 Compare All Three Methods

In [ ]:
print(f"{'QID':>4}  {'Dense':>6} {'BM25':>6} {'Hybrid':>7}")
print("-" * 30)
for q in questions[:N_QUESTIONS]:
    qid = q["id"]
    d = dense_preds.get(qid, "-")
    b = bm25_preds.get(qid, "-")
    h = hybrid_preds.get(qid, "-")
    match = "" if d == b == h else "  ← differ"
    print(f"Q{qid:>3}  {d:>6} {b:>6} {h:>7}{match}")

### Write Submission

Pick your best method and generate a `submission.csv` for Kaggle.

> Set `N_QUESTIONS = 100` at the top and re-run the notebook to generate a full submission.

In [ ]:
# Use dense predictions as the submission (change to hybrid_preds or bm25_preds to try others)
best_preds = dense_preds

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        writer.writerow([qid, best_preds.get(qid, 1)])  # default=1 for unanswered

print(f"submission.csv written ({len(questions)} rows)")

---
## Section 4: What's Next?

This baseline uses a simple setup. Here are ideas to improve your score:

- **Better embeddings** — try other, stronger multilingual  embedding.
- **Smarter chunking** — split by structure or other methods or add useful information to each chunk
- **Chunk size tuning** — experiment with  256, 512, 1024 or something else character chunks
- **Different ThaiLLMs** — try `openthaigpt`, `kbtg`, `pathumma`.
- **Prompt engineering** — adjust the system prompt, add few-shot examples, or change the output format
- **Reranking** — use a cross-encoder or specialized reranker to re-score the top-k chunks before sending to the

**Feel free to implement your own RAG.**